# 04 — Transcriptome Overlay

**Goal:** Are MROH6 copies expressed in song-control nuclei (HVC, RA)?

**Data source:** Colquitt et al. 2021 (Science, doi:10.1126/science.abd9704)  
**GEO accession:** GSE148997

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from pathlib import Path
import subprocess
import os

DATA_TRANS = Path('../data/transcriptome/colquitt_2021')
RESULTS = Path('../results')
DATA_TRANS.mkdir(parents=True, exist_ok=True)

sc.settings.verbosity = 2
sc.set_figure_params(dpi=100, frameon=False)
sns.set_context('notebook')

ImportError: Numba needs NumPy 2.3 or less. Got NumPy 2.4.

## 4a. Download Colquitt et al. 2021 data

The dataset includes scRNA-seq from zebra finch song nuclei (HVC, RA, Area X).

In [ ]:
# Check if data already downloaded
# The GEO page for GSE148997 has processed count matrices
# We need to download the supplementary files

geo_id = 'GSE148997'

# Check for existing files
existing = list(DATA_TRANS.glob('*'))
if existing:
    print(f"Found existing files in {DATA_TRANS}:")
    for f in existing:
        print(f"  {f.name}")
else:
    print(f"No data found. Downloading from GEO ({geo_id})...")
    print("\nOption 1: Use GEOquery/wget for supplementary files")
    print("Option 2: Use the processed h5ad if available")
    print("\nAttempting download...")
    
    # Download via GEO FTP
    # The Colquitt 2021 paper typically provides count matrices as supplementary
    result = subprocess.run(
        ['wget', '-r', '-np', '-nd', '-P', str(DATA_TRANS),
         f'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE148nnn/{geo_id}/suppl/'],
        capture_output=True, text=True, timeout=600
    )
    if result.returncode == 0:
        print("Download complete.")
    else:
        print(f"wget failed. Try manual download from:")
        print(f"  https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={geo_id}")
        print(f"Place files in: {DATA_TRANS}")

: 

In [ ]:
# List downloaded files
print("Files in transcriptome directory:")
for f in sorted(DATA_TRANS.glob('*')):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} ({size_mb:.1f} MB)")

: 

## 4b. Load and preprocess with scanpy

In [ ]:
# Load data — adapt based on actual file format
# Common formats: h5ad, mtx+barcodes+features, csv

h5ad_files = list(DATA_TRANS.glob('*.h5ad'))
mtx_files = list(DATA_TRANS.glob('*.mtx*'))
csv_files = list(DATA_TRANS.glob('*.csv*'))
h5_files = list(DATA_TRANS.glob('*.h5'))

adata = None

if h5ad_files:
    print(f"Loading h5ad: {h5ad_files[0].name}")
    adata = sc.read_h5ad(h5ad_files[0])
elif h5_files:
    print(f"Loading h5: {h5_files[0].name}")
    adata = sc.read_10x_h5(h5_files[0])
elif mtx_files:
    print(f"Loading mtx files from {DATA_TRANS}")
    adata = sc.read_10x_mtx(DATA_TRANS)
elif csv_files:
    print(f"Loading CSV: {csv_files[0].name}")
    df = pd.read_csv(csv_files[0], index_col=0)
    adata = sc.AnnData(df)
else:
    # Try loading from compressed files
    gz_files = list(DATA_TRANS.glob('*.gz'))
    if gz_files:
        print(f"Found compressed files: {[f.name for f in gz_files]}")
        print("Decompressing...")
        for gz in gz_files:
            subprocess.run(['gunzip', '-k', str(gz)], capture_output=True)
        # Retry detection
        mtx_files = list(DATA_TRANS.glob('*.mtx'))
        if mtx_files:
            adata = sc.read_10x_mtx(DATA_TRANS)

if adata is not None:
    print(f"\nLoaded AnnData: {adata.shape[0]} cells x {adata.shape[1]} genes")
    print(f"Obs columns: {list(adata.obs.columns)[:10]}")
    print(f"Var columns: {list(adata.var.columns)[:10]}")
else:
    print("Could not load data automatically.")
    print("Please check the downloaded files and load manually.")

: 

In [ ]:
# Standard preprocessing (if not already processed)
if adata is not None:
    # Check if already normalized
    if adata.X.max() > 100:  # likely raw counts
        print("Preprocessing raw counts...")
        adata.var_names_make_unique()
        sc.pp.filter_cells(adata, min_genes=200)
        sc.pp.filter_genes(adata, min_cells=3)
        adata.raw = adata.copy()
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=2000)
        sc.pp.pca(adata)
        sc.pp.neighbors(adata)
        sc.tl.umap(adata)
        sc.tl.leiden(adata)
        print("Preprocessing complete.")
    else:
        print("Data appears already normalized.")
        if 'X_umap' not in adata.obsm:
            print("Computing UMAP...")
            sc.pp.pca(adata)
            sc.pp.neighbors(adata)
            sc.tl.umap(adata)

: 

In [ ]:
# Search for MROH6 in feature list
if adata is not None:
    gene_names = list(adata.var_names)
    
    # Search various possible names
    search_terms = ['MROH6', 'mroh6', 'Mroh6', 'MRHOH6',
                    'LOC', 'maestro']
    
    found_genes = []
    for term in search_terms:
        matches = [g for g in gene_names if term.lower() in g.lower()]
        if matches:
            found_genes.extend(matches)
            print(f"Matches for '{term}': {matches[:10]}")
    
    if not found_genes:
        print("MROH6 not found by name. Will need sequence-based search.")
        print(f"\nSample gene names: {gene_names[:20]}")
    else:
        found_genes = list(set(found_genes))
        print(f"\nGenes to examine: {found_genes}")

: 

## 4c. Visualize MROH6 expression

In [ ]:
# Plot MROH6 expression if found
if adata is not None and found_genes:
    mroh6_gene = found_genes[0]  # use first match
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # UMAP colored by MROH6 expression
    sc.pl.umap(adata, color=mroh6_gene, ax=axes[0], show=False,
               title=f'{mroh6_gene} expression', cmap='viridis')
    
    # UMAP colored by cluster/region
    region_col = None
    for col in ['brain_region', 'region', 'tissue', 'cluster', 'leiden']:
        if col in adata.obs.columns:
            region_col = col
            break
    if region_col:
        sc.pl.umap(adata, color=region_col, ax=axes[1], show=False,
                   title=f'Colored by {region_col}')
    
    plt.tight_layout()
    plt.savefig(RESULTS / 'figures' / 'mroh6_expression_umap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Skipping visualization (data or gene not available).")

: 

In [ ]:
# Violin plots by brain region and cell type
if adata is not None and found_genes:
    plot_genes = found_genes[:5]  # limit to 5
    
    # By brain region
    if region_col:
        sc.pl.violin(adata, plot_genes, groupby=region_col,
                     rotation=45, stripplot=False)
        plt.savefig(RESULTS / 'figures' / 'mroh6_violin_region.png', dpi=150, bbox_inches='tight')
        plt.show()
    
    # By cell type if available
    celltype_col = None
    for col in ['cell_type', 'celltype', 'type', 'annotation']:
        if col in adata.obs.columns:
            celltype_col = col
            break
    
    if celltype_col:
        sc.pl.violin(adata, plot_genes, groupby=celltype_col,
                     rotation=45, stripplot=False)
        plt.savefig(RESULTS / 'figures' / 'mroh6_violin_celltype.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("Skipping violin plots.")

: 

In [ ]:
# Dotplot: MROH6 vs other multicopy gene families
if adata is not None and found_genes and region_col:
    # Other known multicopy families in birds for comparison
    comparison_genes = []
    candidates = ['MROH6', 'MROH1', 'MROH2', 'MROH3', 'MROH4', 'MROH5',
                  'MROH7', 'MROH8', 'MROH9',
                  'GAPDH', 'ACTB', 'FOXP2', 'BDNF',  # control genes
                  'LSS']  # neighboring gene
    
    for cand in candidates:
        matches = [g for g in gene_names if cand.lower() in g.lower()]
        if matches:
            comparison_genes.append(matches[0])
    
    if comparison_genes:
        all_plot_genes = list(set(found_genes + comparison_genes))
        sc.pl.dotplot(adata, all_plot_genes, groupby=region_col)
        plt.savefig(RESULTS / 'figures' / 'mroh6_dotplot.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("Skipping dotplot.")

: 

In [ ]:
# Summary
print("="*60)
print("TRANSCRIPTOME OVERLAY SUMMARY")
print("="*60)
if adata is not None:
    print(f"Dataset: Colquitt et al. 2021 ({geo_id})")
    print(f"Cells: {adata.shape[0]}, Genes: {adata.shape[1]}")
    if found_genes:
        print(f"MROH6 found as: {found_genes}")
        # Calculate basic expression stats
        for gene in found_genes[:3]:
            if gene in adata.var_names:
                expr = adata[:, gene].X
                if hasattr(expr, 'toarray'):
                    expr = expr.toarray().flatten()
                pct_expressing = (expr > 0).mean() * 100
                print(f"  {gene}: {pct_expressing:.1f}% of cells express")
    else:
        print("MROH6 NOT found in feature list.")
        print("Possible reasons:")
        print("  - Gene annotation uses different name")
        print("  - MROH6 copies not annotated as genes")
        print("  - Reads map ambiguously across copies")
else:
    print("Data not loaded. Download from GEO manually if needed.")

: 